# Mohan Sir workshop twin pilot — safe preflight

This notebook checks Colab GPU availability and resolves the seven user-provided Bunny Stream **source recordings**. They are outfit/look inputs, not generated twins. The intended later pilot is to lip-sync approved new workshop speech to the matching recording, preserving the outfit already in that footage.

**This notebook does not install models, download checkpoints, run inference, launch a worker, upload files, or claim acceptance.** MuseTalk inference remains blocked pending the outstanding artifact/dependency/training-data review documented in `docs/MUSETALK_GPU_ACCEPTANCE_REPORT.md` and an authorized GPU session. Source downloads are optional and require an explicit confirmation below. Do not use footage or audio unless you have authorization and consent for this processing.

## 1. Check the current Colab runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another authorized NVIDIA GPU), then run this cell. A GPU is not guaranteed. This only inspects the runtime; it does not install packages.

In [ ]:
import importlib.util
import platform
import subprocess

print("Python:", platform.python_version())
if importlib.util.find_spec("torch") is None:
    print("PyTorch: not installed (expected for this preflight; nothing was installed).")
else:
    import torch
    print("PyTorch:", torch.__version__)
    print("PyTorch CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA runtime:", torch.version.cuda)

try:
    result = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True, timeout=15)
    print(result.stdout if result.returncode == 0 else "nvidia-smi unavailable:", result.stderr[:1000])
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print("No NVIDIA GPU was detected:", str(exc))

## 2. Source recordings

The notebook uses Bunny's authenticated Stream API to check metadata. Add `BUNNY_STREAM_API_KEY` and `BUNNY_STREAM_LIBRARY_ID` in the Colab **Secrets** panel (key icon), enable notebook access, and never paste either value into a cell or share its output. The library ID is included in each supplied player link; the API key is private.

The CDN hostname is read from the authenticated library metadata response. If the account/API does not return it, stop and resolve that with Bunny rather than guessing a host.

In [ ]:
from urllib.parse import urlparse

SOURCES = [
    {"look": "Paret-Popat / lime-green kurta", "player_url": "https://player.mediadelivery.net/play/606386/5fd3882c-d768-4335-a33e-229a057f37d7-popat"},
    {"look": "Golden / cream kurta", "player_url": "https://player.mediadelivery.net/play/606386/c828f318-88d6-48e4-b0c9-fe5004ae59f0"},
    {"look": "White kurta", "player_url": "https://player.mediadelivery.net/play/606386/c16ba961-5625-47bf-9193-dbec875673a7"},
    {"look": "White kurta 2", "player_url": "https://player.mediadelivery.net/play/606386/247e9982-7723-49bb-a327-d923c67e543a"},
    {"look": "Modi jacket", "player_url": "https://player.mediadelivery.net/play/606386/f41134e9-ac7a-4495-98d9-27cc7a3f4784"},
    {"look": "White shirt", "player_url": "https://player.mediadelivery.net/play/606386/33b599f9-fbfc-432e-9339-b5f7b4a0e5b3"},
    {"look": "Golden kurta 2", "player_url": "https://player.mediadelivery.net/play/606386/d06767ad-3ba3-4695-b3e5-32e3012fdbc2"},
]

def parse_player_url(player_url):
    parsed = urlparse(player_url)
    parts = [part for part in parsed.path.split("/") if part]
    if parsed.scheme != "https" or parsed.hostname != "player.mediadelivery.net" or len(parts) != 3 or parts[0] != "play":
        raise ValueError("Unexpected Bunny player URL; refusing to derive an API identifier: " + player_url)
    library_id, video_guid = parts[1], parts[2]
    if not library_id.isdigit() or not video_guid or any(ch in video_guid for ch in "?#/"):
        raise ValueError("Invalid library/video identifier in: " + player_url)
    return library_id, video_guid

for source in SOURCES:
    source["library_id"], source["video_guid"] = parse_player_url(source["player_url"])
    print(source["look"], "— library", source["library_id"], "— video", source["video_guid"])

library_ids = {source["library_id"] for source in SOURCES}
if len(library_ids) != 1:
    raise RuntimeError("The supplied links do not all use one Bunny Stream library; inspect before continuing.")

In [ ]:
import json
import urllib.error
import urllib.request

try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab and configure its Secrets panel first.") from exc

library_id = next(iter(library_ids))
api_key = userdata.get("BUNNY_STREAM_API_KEY")
configured_library_id = userdata.get("BUNNY_STREAM_LIBRARY_ID")
if not api_key or not configured_library_id:
    raise RuntimeError("Set BUNNY_STREAM_API_KEY and BUNNY_STREAM_LIBRARY_ID in Colab Secrets; values are never printed.")
if str(configured_library_id) != library_id:
    raise RuntimeError("BUNNY_STREAM_LIBRARY_ID does not match the library in the supplied URLs.")

def bunny_get_json(path):
    request = urllib.request.Request(
        "https://video.bunnycdn.com" + path,
        headers={"AccessKey": api_key, "Accept": "application/json"},
        method="GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            if response.status != 200:
                raise RuntimeError("Bunny API returned HTTP " + str(response.status))
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        raise RuntimeError("Bunny API request failed with HTTP " + str(exc.code) + "; verify library ID and API-key permissions.") from None

library = bunny_get_json("/library/" + library_id)
cdn_host = library.get("hostname") or library.get("Hostname")
if not isinstance(cdn_host, str) or not cdn_host.strip():
    raise RuntimeError("Bunny did not return a CDN hostname. Stop; do not guess the media host.")
cdn_host = cdn_host.strip().lower().rstrip("/")
if "://" in cdn_host or "/" in cdn_host or "@" in cdn_host:
    raise RuntimeError("Unexpected CDN hostname format; refusing to construct media URLs.")
print("Bunny Stream library authenticated. CDN hostname:", cdn_host)
print("API key was not displayed.")

In [ ]:
def normalized_value(data, *names):
    lowered = {str(key).lower(): value for key, value in data.items()}
    for name in names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    return None

for source in SOURCES:
    try:
        metadata = bunny_get_json("/library/" + source["library_id"] + "/videos/" + source["video_guid"])
        source["metadata"] = metadata
        resolutions = normalized_value(metadata, "availableResolutions")
        status = normalized_value(metadata, "status")
        source["available_resolutions"] = [item.strip() for item in resolutions.split(",") if item.strip()] if isinstance(resolutions, str) else []
        print(json.dumps({
            "look": source["look"],
            "guid": source["video_guid"],
            "title": normalized_value(metadata, "title"),
            "status": status,
            "length_seconds": normalized_value(metadata, "length"),
            "available_resolutions": source["available_resolutions"],
            "is_public": normalized_value(metadata, "isPublic"),
            "has_mp4_fallback": normalized_value(metadata, "hasMP4Fallback"),
        }, ensure_ascii=False))
    except (RuntimeError, ValueError) as exc:
        source["metadata_error"] = str(exc)
        print(json.dumps({"look": source["look"], "guid": source["video_guid"], "error": str(exc)}, ensure_ascii=False))

print("A metadata error is a blocker for that source. Do not guess or silently replace its video ID.")

## 3. Optional source MP4 download

Only proceed if Bunny metadata succeeded, shows an available resolution and confirms the recording is public/playable under your account. This cell asks you to type `DOWNLOAD` and choose exactly one recording. It streams that rendition to the temporary Colab runtime, checks the response and file size, and calculates SHA-256. Colab storage is temporary. No file is uploaded or shared. If Bunny's library response, encoding status, or CDN behavior differs from expectations, stop and resolve it with Bunny rather than altering the URL or bypassing access controls.

In [ ]:
import hashlib
import os
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen

confirmation = input('Type DOWNLOAD to fetch one source MP4 into this temporary Colab runtime: ').strip()
if confirmation != "DOWNLOAD":
    print("Skipped. No source media was downloaded.")
else:
    eligible = [
        source for source in SOURCES
        if "metadata" in source
        and not source.get("metadata_error")
        and source.get("available_resolutions")
        and normalized_value(source["metadata"], "isPublic") is True
    ]
    if not eligible:
        raise RuntimeError("No source has successful metadata, an available resolution, and public access; nothing downloaded.")
    for index, source in enumerate(eligible, start=1):
        print(index, source["look"], "—", source["available_resolutions"])
    choice = int(input("Enter the number of exactly one recording to download: ").strip())
    if choice < 1 or choice > len(eligible):
        raise ValueError("Selection is outside the displayed source list.")
    selected = eligible[choice - 1]
    resolution = input("Enter one exact available resolution shown above (for example 720p): ").strip()
    if resolution not in selected["available_resolutions"]:
        raise ValueError("Resolution is not listed by Bunny metadata; nothing downloaded.")
    media_url = "https://" + cdn_host + "/" + quote(selected["video_guid"], safe="-") + "/play_" + quote(resolution, safe="") + ".mp4"
    parsed_media = urlparse(media_url)
    if parsed_media.scheme != "https" or parsed_media.hostname != cdn_host:
        raise RuntimeError("Constructed media URL failed the HTTPS/host check.")
    output_dir = Path("/content/mohan_twin_source_preflight")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / (selected["video_guid"] + "_" + resolution + ".mp4")
    request = Request(media_url, headers={"Range": "bytes=0-"}, method="GET")
    digest = hashlib.sha256()
    bytes_written = 0
    try:
        with urlopen(request, timeout=60) as response, output_path.open("wb") as output:
            if response.status not in (200, 206):
                raise RuntimeError("CDN returned HTTP " + str(response.status) + "; no result accepted.")
            final_url = urlparse(response.geturl())
            if final_url.scheme != "https" or final_url.hostname != cdn_host:
                raise RuntimeError("CDN redirected outside the configured HTTPS host; refusing the download.")
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                digest.update(chunk)
                bytes_written += len(chunk)
    except Exception:
        if output_path.exists():
            output_path.unlink()
        raise
    if bytes_written == 0 or not output_path.is_file():
        output_path.unlink(missing_ok=True)
        raise RuntimeError("Downloaded file is empty; it is not an accepted source.")
    print(json.dumps({
        "look": selected["look"],
        "file": str(output_path),
        "bytes": bytes_written,
        "sha256": digest.hexdigest(),
        "note": "Source recording only; not an AI-generated twin.",
    }, ensure_ascii=False, indent=2))

## Stop here and report the preflight evidence

Before any future inference implementation, report:

1. Whether Colab has an authorized NVIDIA GPU; its model name and CUDA/PyTorch status (if already installed).
2. The metadata result for each source and any failing video GUID. Do not share the API key.
3. For any explicitly downloaded source: look label, file size, and SHA-256. Transfer the file only through an approved private channel; do not post source footage publicly.
4. Which source is authorized for the pilot, the approved workshop script/audio and language, and confirmation that the person/rights holder consented to this lip-sync processing.

No output video exists from this preflight. Model package installation, checkpoint retrieval, inference, Bunny output upload/read-back, and production integration remain disabled pending the documented license/provenance review and a separately approved implementation step.